In [1]:
# === CONFIGURATION (ONLY SECTION THAT DIFFERS ACROSS NOTEBOOKS) ===
NUM_TASKS = 1                          # 1 = single-task (LogD only)
PRETRAINED_CHECKPOINT = None            # None = random init
EXPERIMENT_NAME = "st_random"


In [2]:
# === IMPORTS + CONSTANTS (identical across all notebooks) ===

import os
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from sklearn.metrics import r2_score
from scipy.stats import spearmanr, kendalltau

from gt_pyg import GraphTransformerNet, get_tensor_data
from gt_pyg.data import get_atom_feature_dim, get_bond_feature_dim

warnings.filterwarnings('ignore', category=FutureWarning)

# --- Reproducibility ---
SEED = 1928374650

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

# --- Hyperparameters ---
BATCH_TRAIN = 256
BATCH_EVAL = 1024
EPOCHS = 1500                # production (set to 10 for quick test)
BASE_LR = 1e-3
MIN_LR = 1e-5                # BASE_LR / 100
WEIGHT_DECAY = 1e-5
WARMUP_EPOCHS = 25
GRAD_CLIP_NORM = 5.0
DELTA = 0.2                  # clipping margin beyond train range
T_MAX = min(500, EPOCHS)
NUM_WORKERS = 4

# Loss weights
W_RAE = 1.0
W_HUBER = 0.25
W_CORR = 0.25
W_TAU = 0.1
W_R2 = 0.1
HUBER_DELTA = 0.5
TAU_TEMP = 2.0
RANK_PAIRS = 512

SPLIT_FRACTIONS = [0.7, 0.2, 0.1]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'NUM_TASKS: {NUM_TASKS}')
print(f'PRETRAINED_CHECKPOINT: {PRETRAINED_CHECKPOINT}')
print(f'EXPERIMENT_NAME: {EXPERIMENT_NAME}')

Device: cpu
NUM_TASKS: 1
PRETRAINED_CHECKPOINT: None
EXPERIMENT_NAME: st_random


In [3]:
# === DATA LOADING ===

TRAIN_CSV = '../data/train-set/expansion_log_data_train.csv'
TEST_CSV = '../data/test-set/expansion_data_test_full_lb_flag.csv'

log_train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# All 9 log-scale endpoints
ALL_LOG_ENDPOINTS = [
    'LogD', 'LogS', 'Log_HLM_CLint', 'Log_MLM_CLint',
    'Log_Caco_Papp_AB', 'Log_Caco_ER',
    'Log_Mouse_PPB', 'Log_Mouse_BPB', 'Log_Mouse_MPB',
]

# Select endpoints based on NUM_TASKS
if NUM_TASKS == 1:
    ENDPOINTS = ['LogD']
else:
    ENDPOINTS = ALL_LOG_ENDPOINTS
    assert NUM_TASKS == len(ENDPOINTS)

print(f'Training data: {len(log_train_df)} molecules, {len(ENDPOINTS)} endpoints')
print(f'Test data: {len(test_df)} molecules')
print(f'Endpoints: {ENDPOINTS}')

Training data: 5326 molecules, 1 endpoints
Test data: 2282 molecules
Endpoints: ['LogD']


In [4]:
# === LOG TRANSFORMS ===
# v1 transforms matching the pre-computed training CSV.
# Forward:  log10((val + 1) * multiplier)
# Inverse:  (10^val / multiplier) - 1, clipped >= 0

ENDPOINT_MAP = {
    # original_col: (log_col, multiplier)
    'LogD':                          ('LogD', 1),
    'KSOL':                          ('LogS', 1e-6),
    'HLM CLint':                     ('Log_HLM_CLint', 1),
    'MLM CLint':                     ('Log_MLM_CLint', 1),
    'Caco-2 Permeability Papp A>B':  ('Log_Caco_Papp_AB', 1e-6),
    'Caco-2 Permeability Efflux':    ('Log_Caco_ER', 1),
    'MPPB':                          ('Log_Mouse_PPB', 1),
    'MBPB':                          ('Log_Mouse_BPB', 1),
    'MGMB':                          ('Log_Mouse_MPB', 1),
}

# Build reverse map: log_col -> (original_col, multiplier)
REVERSE_MAP = {v[0]: (k, v[1]) for k, v in ENDPOINT_MAP.items()}


def log_transform_test_data(test_df, endpoints):
    """Transform test data from original scale to log scale."""
    result = pd.DataFrame()
    result['SMILES'] = test_df['SMILES']
    result['Molecule Name'] = test_df['Molecule Name']
    for log_col in endpoints:
        orig_col, mult = REVERSE_MAP[log_col]
        if orig_col in test_df.columns:
            vals = test_df[orig_col].values.astype(float)
            if log_col == 'LogD':  # already log-scale
                result[log_col] = vals
            else:
                result[log_col] = np.log10((vals + 1) * mult)
        else:
            result[log_col] = np.nan
    if 'is_leaderboard' in test_df.columns:
        result['is_leaderboard'] = test_df['is_leaderboard'].values
    return result


def inverse_log_transform(preds_df, endpoints):
    """Transform predictions from log scale back to original scale."""
    result = preds_df.copy()
    for log_col in endpoints:
        orig_col, mult = REVERSE_MAP[log_col]
        if log_col in result.columns:
            vals = result[log_col].values.astype(float)
            if log_col == 'LogD':  # already original-ish scale
                pass  # no transform needed
            else:
                result[log_col] = np.maximum((10 ** vals / mult) - 1, 0)
    return result


# Transform test data to log scale for training/evaluation
log_test_df = log_transform_test_data(test_df, ENDPOINTS)
print(f'Log-transformed test data: {len(log_test_df)} rows')


Log-transformed test data: 2282 rows


In [5]:
# === DATA SPLITTING + PyG DATASETS ===

def split_data(n, fractions, seed=SEED):
    """Random split into subsets. Returns index arrays."""
    rng = np.random.RandomState(seed)
    indices = rng.permutation(n)
    splits = []
    start = 0
    for i, frac in enumerate(fractions):
        if i == len(fractions) - 1:
            splits.append(indices[start:])
        else:
            end = start + int(n * frac)
            splits.append(indices[start:end])
            start = end
    return tuple(splits)


# Split train CSV into train/val/test
n = len(log_train_df)
train_idx, val_idx, test_from_train_idx = split_data(n, SPLIT_FRACTIONS)

train_df = log_train_df.iloc[train_idx].reset_index(drop=True)
val_df = log_train_df.iloc[val_idx].reset_index(drop=True)
test_from_train_df = log_train_df.iloc[test_from_train_idx].reset_index(drop=True)

print(f'Split: train={len(train_df)}  val={len(val_df)}  test_from_train={len(test_from_train_df)}')

# Build PyG datasets
def build_dataset(df, endpoints):
    smiles = df['SMILES'].astype(str).tolist()
    Y = df[list(endpoints)].to_numpy(dtype=float).tolist()
    return get_tensor_data(smiles, Y, gnm=True)


print('Building datasets...')
train_data = build_dataset(train_df, ENDPOINTS)
val_data = build_dataset(val_df, ENDPOINTS)
test_from_train_data = build_dataset(test_from_train_df, ENDPOINTS)

# Also build external test set (log-transformed)
ext_test_data = build_dataset(log_test_df, ENDPOINTS)

train_loader = DataLoader(train_data, batch_size=BATCH_TRAIN, shuffle=True, num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_data, batch_size=BATCH_EVAL, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0)
test_from_train_loader = DataLoader(test_from_train_data, batch_size=BATCH_EVAL, shuffle=False)
ext_test_loader = DataLoader(ext_test_data, batch_size=BATCH_EVAL, shuffle=False)

print('Datasets built.')


Split: train=3728  val=1065  test_from_train=533
Building datasets...


Processing data:   0%|          | 0/3728 [00:00<?, ?it/s]

Processing data:   0%|          | 0/1065 [00:00<?, ?it/s]

Processing data:   0%|          | 0/533 [00:00<?, ?it/s]

Processing data:   0%|          | 0/2282 [00:00<?, ?it/s]

Datasets built.


In [6]:
# === MODEL CONSTRUCTION ===

model = GraphTransformerNet(
    node_dim_in=get_atom_feature_dim(),
    edge_dim_in=get_bond_feature_dim(),
    num_gt_layers=4,
    hidden_dim=128,
    num_heads=8,
    norm='bn',
    gt_aggregators=['sum', 'mean'],
    aggregators=['sum', 'mean', 'max', 'std'],
    dropout=0.3,
    act='gelu',
    gate=True,
    qkv_bias=False,
    num_tasks=NUM_TASKS,
).to(device)

n_params = model.num_parameters()
print(f'Model: {n_params:,} trainable parameters, num_tasks={NUM_TASKS}')

Model: 2,797,634 trainable parameters, num_tasks=1


In [7]:
# === PRETRAINED WEIGHT LOADING (shape-filter pattern) ===
# BUG FIX: PyTorch strict=False does NOT handle shape mismatches —
# it only handles missing/extra keys. We must filter by shape.

if PRETRAINED_CHECKPOINT is not None:
    ckpt_path = Path(PRETRAINED_CHECKPOINT)
    if not ckpt_path.exists():
        print(f'WARNING: Checkpoint not found at {ckpt_path} — using random init')
    else:
        from gt_pyg.nn import load_checkpoint
        checkpoint = load_checkpoint(ckpt_path, map_location=device)
        pretrain_sd = checkpoint['model_state_dict']
        finetune_sd = model.state_dict()

        compatible_sd = {
            k: v for k, v in pretrain_sd.items()
            if k in finetune_sd and v.shape == finetune_sd[k].shape
        }
        skipped = [
            k for k in pretrain_sd
            if k not in finetune_sd or pretrain_sd[k].shape != finetune_sd.get(k, torch.tensor([])).shape
        ]

        model.load_state_dict(compatible_sd, strict=False)
        print(f'Loaded {len(compatible_sd)}/{len(pretrain_sd)} pretrained keys')
        if skipped:
            print(f'Skipped (shape mismatch or missing): {skipped}')
else:
    print('Random initialization (no pretrained checkpoint)')

Random initialization (no pretrained checkpoint)


In [8]:
# === 5-COMPONENT CUSTOM LOSS ===


@torch.no_grad()
def compute_task_scales(loader, num_tasks, eps=1e-8):
    """MAD per task from training data, for RAE normalization."""
    ys = [[] for _ in range(num_tasks)]
    for batch in loader:
        y = batch.y.view(-1, num_tasks).cpu().numpy()
        m = batch.y_mask.view(-1, num_tasks).cpu().numpy().astype(bool)
        for t in range(num_tasks):
            yt = y[m[:, t], t]
            yt = yt[np.isfinite(yt)]
            if yt.size:
                ys[t].append(yt)
    scales = []
    for t in range(num_tasks):
        if not ys[t]:
            scales.append(1.0)
            continue
        v = np.concatenate(ys[t])
        if v.size < 3:
            scales.append(1.0)
            continue
        med = np.median(v)
        mad = np.median(np.abs(v - med))
        scales.append(float(max(mad, eps)))
    return torch.tensor(scales, dtype=torch.float32)


def masked_weighted_rae_loss(pred, y, mask, task_scale, eps=1e-8, clip_val=100.0):
    """Relative absolute error normalized by per-task MAD."""
    pred = torch.clamp(pred, -clip_val, clip_val)
    valid = (mask > 0) & torch.isfinite(y) & torch.isfinite(pred)
    diff = torch.where(valid, (pred - y).abs() / (task_scale.to(pred.device) + eps), torch.zeros_like(pred))
    w = torch.where(valid, torch.ones_like(pred), torch.zeros_like(pred))
    sum_t = (diff * w).sum(dim=0)
    sum_w = w.sum(dim=0).clamp_min(eps)
    mean_t = sum_t / sum_w
    task_mask = w.sum(dim=0) > 0
    return mean_t[task_mask].mean() if task_mask.any() else pred.new_tensor(0.0)


def masked_weighted_huber_loss(pred, y, mask, delta=1.0, task_scale=None, clip_val=100.0, eps=1e-8):
    """Huber loss, optionally normalized by task scale."""
    pred = torch.clamp(pred, -clip_val, clip_val)
    valid = (mask > 0) & torch.isfinite(y) & torch.isfinite(pred)
    diff = torch.where(valid, pred - y, torch.zeros_like(pred))
    if task_scale is not None:
        diff = diff / (task_scale.to(diff.device) + eps)
    abs_diff = diff.abs()
    quad = torch.minimum(abs_diff, abs_diff.new_tensor(delta))
    loss = 0.5 * quad ** 2 + delta * (abs_diff - quad)
    w = torch.where(valid, torch.ones_like(pred), torch.zeros_like(pred))
    sum_t = (loss * w).sum(dim=0)
    sum_w = w.sum(dim=0).clamp_min(eps)
    mean_t = sum_t / sum_w
    task_mask = w.sum(dim=0) > 0
    return mean_t[task_mask].mean() if task_mask.any() else pred.new_tensor(0.0)


def masked_weighted_corr_loss(pred, y, mask, eps=1e-8, clip_val=100.0):
    """1 - Pearson correlation per task."""
    pred = torch.clamp(pred, -clip_val, clip_val)
    valid = (mask > 0) & torch.isfinite(y) & torch.isfinite(pred)
    w = torch.where(valid, torch.ones_like(pred), torch.zeros_like(pred))
    sum_w = w.sum(dim=0).clamp_min(eps)
    p_v = torch.where(valid, pred, torch.zeros_like(pred))
    y_v = torch.where(valid, y, torch.zeros_like(y))
    mean_p = (w * p_v).sum(dim=0) / sum_w
    mean_y = (w * y_v).sum(dim=0) / sum_w
    p_c = torch.where(valid, p_v - mean_p.unsqueeze(0), torch.zeros_like(pred))
    y_c = torch.where(valid, y_v - mean_y.unsqueeze(0), torch.zeros_like(y))
    cov = (w * p_c * y_c).sum(dim=0)
    std_p = torch.sqrt((w * p_c ** 2).sum(dim=0) + eps)
    std_y = torch.sqrt((w * y_c ** 2).sum(dim=0) + eps)
    corr = cov / (std_p * std_y + eps)
    per_task = 1.0 - corr
    task_mask = w.sum(dim=0) > 0
    return per_task[task_mask].mean() if task_mask.any() else pred.new_tensor(0.0)


def masked_weighted_kendall_rank_loss(pred, y, mask, num_pairs=512, tau_temp=1.0, clip_val=100.0, eps=1e-8):
    """Differentiable Kendall rank loss via sampled pairs."""
    pred = torch.clamp(pred, -clip_val, clip_val)
    B, T = pred.shape
    mask_b = mask.bool()
    per_task_loss = []
    for t in range(T):
        valid_t = mask_b[:, t] & torch.isfinite(y[:, t]) & torch.isfinite(pred[:, t])
        idx = torch.where(valid_t)[0]
        n = idx.numel()
        if n < 2:
            per_task_loss.append(pred.new_tensor(0.0))
            continue
        max_pairs = n * (n - 1) // 2
        ii, jj = torch.triu_indices(n, n, offset=1, device=pred.device)
        if max_pairs > num_pairs:
            probe = min(max_pairs, 8192)
            choose = torch.randperm(max_pairs, device=pred.device)[:probe]
            ia, ib = ii[choose], jj[choose]
            ydiff = (y[idx[ia], t] - y[idx[ib], t]).abs()
            topk = torch.topk(ydiff, k=min(num_pairs, probe)).indices
            ia, ib = ia[topk], ib[topk]
        else:
            ia, ib = ii, jj
        a, b = idx[ia], idx[ib]
        s = torch.sign(y[a, t] - y[b, t])
        p_diff = pred[a, t] - pred[b, t]
        non_tie = s != 0
        if non_tie.any():
            per_task_loss.append(F.softplus(-s[non_tie] * p_diff[non_tie] / tau_temp).mean())
        else:
            per_task_loss.append(pred.new_tensor(0.0))
    losses = torch.stack(per_task_loss)
    return losses.mean()


def masked_r2_style_loss(pred, y, mask, eps=1e-8, clip_val=100.0):
    """R2-style loss: SSE / Var(y) per task."""
    pred = torch.clamp(pred, -clip_val, clip_val)
    valid = mask.bool() & torch.isfinite(y) & torch.isfinite(pred)
    counts = valid.sum(dim=0)
    has_data = counts > 1
    if not has_data.any():
        return pred.new_tensor(0.0)
    p_v = torch.where(valid, pred, torch.zeros_like(pred))
    y_v = torch.where(valid, y, torch.zeros_like(y))
    mean_y = y_v.sum(dim=0) / (counts + eps)
    y_c = torch.where(valid, y - mean_y.unsqueeze(0), torch.zeros_like(y))
    sse = ((p_v - y_v) ** 2).sum(dim=0)
    var_y = (y_c ** 2).sum(dim=0)
    good = has_data & (var_y > eps)
    if not good.any():
        return pred.new_tensor(0.0)
    return (sse[good] / (var_y[good] + eps)).mean()


def custom_loss(pred, y, mask, *, task_scale):
    """Combined 5-component loss."""
    L = pred.new_tensor(0.0)
    if W_RAE > 0:
        L = L + W_RAE * masked_weighted_rae_loss(pred, y, mask, task_scale)
    if W_HUBER > 0:
        L = L + W_HUBER * masked_weighted_huber_loss(pred, y, mask, delta=HUBER_DELTA, task_scale=task_scale)
    if W_CORR > 0:
        L = L + W_CORR * masked_weighted_corr_loss(pred, y, mask)
    if W_TAU > 0:
        L = L + W_TAU * masked_weighted_kendall_rank_loss(pred, y, mask, num_pairs=RANK_PAIRS, tau_temp=TAU_TEMP)
    if W_R2 > 0:
        L = L + W_R2 * masked_r2_style_loss(pred, y, mask)
    return L


# Compute task scales from training data
task_scale = compute_task_scales(train_loader, NUM_TASKS).to(device)
print(f'Task scales (MAD): {task_scale.cpu().tolist()}')


Task scales (MAD): [0.7999999523162842]


In [9]:
# === METRICS ===

def official_metrics(y_true, y_pred):
    """Compute MAE, RAE, R2, Spearman, Kendall for 1D arrays."""
    y = np.asarray(y_true).ravel()
    p = np.asarray(y_pred).ravel()
    m = np.isfinite(y) & np.isfinite(p)
    y, p = y[m], p[m]
    if y.size == 0:
        return {'MAE': np.nan, 'RAE': np.nan, 'R2': np.nan, 'Spearman': np.nan, 'Kendall': np.nan}
    mae = float(np.mean(np.abs(y - p)))
    denom = np.mean(np.abs(y - np.mean(y)))
    rae = float(mae / denom) if denom > 0 else np.nan
    r2 = float(r2_score(y, p)) if np.std(y) > 0 else np.nan
    spr = float(spearmanr(y, p).statistic) if np.std(p) > 1e-6 else np.nan
    ktau = float(kendalltau(y, p).statistic) if np.std(p) > 1e-6 else np.nan
    return {'MAE': mae, 'RAE': rae, 'R2': r2, 'Spearman': spr, 'Kendall': ktau}


@torch.no_grad()
def evaluate_loader(model, loader, endpoints):
    """Evaluate model on a loader. Returns (macro_rae, per_endpoint_metrics, preds, targets, masks)."""
    model.eval()
    preds_all, y_all, m_all = [], [], []
    T = len(endpoints)
    for batch in loader:
        batch = batch.to(device)
        out, _ = model(batch.x, batch.edge_index, batch.edge_attr, batch=batch.batch, zero_var=False)
        y = batch.y.view(-1, T)
        mask = batch.y_mask.view(-1, T)
        preds_all.append(out.cpu().numpy())
        y_all.append(y.cpu().numpy())
        m_all.append(mask.cpu().numpy())
    if not preds_all:
        return np.inf, {}, np.zeros((0, T)), np.zeros((0, T)), np.zeros((0, T))
    preds = np.vstack(preds_all)
    ys = np.vstack(y_all)
    masks = np.vstack(m_all)
    metrics = {}
    raes = []
    for i, ep in enumerate(endpoints):
        m = masks[:, i].astype(bool) & np.isfinite(ys[:, i])
        if m.sum() < 3:
            metrics[ep] = {'MAE': np.nan, 'RAE': np.nan, 'R2': np.nan, 'Spearman': np.nan, 'Kendall': np.nan}
            continue
        metrics[ep] = official_metrics(ys[m, i], preds[m, i])
        raes.append(metrics[ep]['RAE'])
    macro_rae = float(np.nanmean(raes)) if raes else np.inf
    return macro_rae, metrics, preds, ys, masks


In [10]:
# === TRAINING LOOP ===

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, T_MAX - WARMUP_EPOCHS)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return MIN_LR / BASE_LR + (1 - MIN_LR / BASE_LR) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Tracking
best_macro_rae = np.inf
best_state = None
best_epoch = 0
history = []

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_losses = []
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out, _ = model(batch.x, batch.edge_index, batch.edge_attr,
                       batch=batch.batch, zero_var=False)
        T = out.size(1)
        y = batch.y.view(-1, T)
        mask = batch.y_mask.view(-1, T)
        valid_mask = mask * (~torch.isnan(y)).float()
        loss = custom_loss(out, y, valid_mask, task_scale=task_scale)
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        train_losses.append(loss.item())

    lr_used = optimizer.param_groups[0]['lr']
    scheduler.step()
    avg_train_loss = np.mean(train_losses) if train_losses else np.nan

    # --- Validate ---
    val_rae, val_metrics, _, _, _ = evaluate_loader(model, val_loader, ENDPOINTS)

    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'val_rae': val_rae, 'lr': lr_used})

    # Best checkpoint by macro RAE
    if val_rae < best_macro_rae:
        best_macro_rae = val_rae
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Logging
    if epoch == 1 or epoch % 25 == 0 or epoch == EPOCHS:
        val_mae = np.nanmean([m.get('MAE', np.nan) for m in val_metrics.values()])
        val_r2 = np.nanmean([m.get('R2', np.nan) for m in val_metrics.values()])
        print(f'Epoch {epoch:4d}/{EPOCHS} | LR={lr_used:.1e} | '
              f'train_loss={avg_train_loss:.4f} | val_RAE={val_rae:.4f} | '
              f'val_MAE={val_mae:.4f} | val_R2={val_r2:.4f} | '
              f'best_RAE={best_macro_rae:.4f}@{best_epoch}')

print(f'\nTraining complete. Best val macro RAE: {best_macro_rae:.4f} at epoch {best_epoch}')


Epoch    1/1500 | LR=4.0e-05 | train_loss=5.2384 | val_RAE=2.2957 | val_MAE=2.1363 | val_R2=-3.1114 | best_RAE=2.2957@1
Epoch   25/1500 | LR=1.0e-03 | train_loss=1.3990 | val_RAE=0.8445 | val_MAE=0.7859 | val_R2=0.3262 | best_RAE=0.7356@23
Epoch   50/1500 | LR=9.9e-04 | train_loss=0.9292 | val_RAE=0.6399 | val_MAE=0.5955 | val_R2=0.6069 | best_RAE=0.5260@48
Epoch   75/1500 | LR=9.7e-04 | train_loss=0.7599 | val_RAE=0.4677 | val_MAE=0.4352 | val_R2=0.7683 | best_RAE=0.4319@72
Epoch  100/1500 | LR=9.4e-04 | train_loss=0.6643 | val_RAE=0.4154 | val_MAE=0.3866 | val_R2=0.8224 | best_RAE=0.3838@94
Epoch  125/1500 | LR=9.0e-04 | train_loss=0.6032 | val_RAE=0.3890 | val_MAE=0.3620 | val_R2=0.8403 | best_RAE=0.3781@114
Epoch  150/1500 | LR=8.4e-04 | train_loss=0.5969 | val_RAE=0.3835 | val_MAE=0.3569 | val_R2=0.8441 | best_RAE=0.3616@147
Epoch  175/1500 | LR=7.8e-04 | train_loss=0.5555 | val_RAE=0.3964 | val_MAE=0.3689 | val_R2=0.8352 | best_RAE=0.3448@166
Epoch  200/1500 | LR=7.1e-04 | train_

In [15]:
# === EVALUATION + PREDICTIONS ===

# Restore best model
if best_state is not None:
    model.load_state_dict(best_state)
model = model.to(device)

# Evaluate on all splits
print('=== Validation set (best model) ===')
val_rae, val_metrics, val_preds, val_ys, val_masks = evaluate_loader(model, val_loader, ENDPOINTS)
for ep, m in val_metrics.items():
    print(f'  {ep:25s}: MAE={m["MAE"]:.4f}  RAE={m["RAE"]:.4f}  R2={m["R2"]:.4f}  rho={m["Spearman"]:.4f}  tau={m["Kendall"]:.4f}')
print(f'  Macro RAE: {val_rae:.4f}')

print('\n=== Test set (from train split, best model) ===')
test_rae, test_metrics, test_preds, test_ys, test_masks = evaluate_loader(model, test_from_train_loader, ENDPOINTS)
for ep, m in test_metrics.items():
    print(f'  {ep:25s}: MAE={m["MAE"]:.4f}  RAE={m["RAE"]:.4f}  R2={m["R2"]:.4f}  rho={m["Spearman"]:.4f}  tau={m["Kendall"]:.4f}')
print(f'  Macro RAE: {test_rae:.4f}')


=== Validation set (best model) ===
  LogD                     : MAE=0.2511  RAE=0.2698  R2=0.9120  rho=0.9538  tau=0.8337
  Macro RAE: 0.2698

=== Test set (from train split, best model) ===
  LogD                     : MAE=0.2395  RAE=0.2546  R2=0.9221  rho=0.9595  tau=0.8469
  Macro RAE: 0.2546


In [16]:
# === EXTERNAL TEST PREDICTIONS ===

# Compute training data range for clipping
train_mins = {}
train_maxs = {}
for ep in ENDPOINTS:
    vals = train_df[ep].dropna().values
    train_mins[ep] = float(vals.min())
    train_maxs[ep] = float(vals.max())

# Predict on external test set
ext_rae, ext_metrics, ext_preds, ext_ys, ext_masks = evaluate_loader(model, ext_test_loader, ENDPOINTS)

# Print unclipped metrics
print('=== External test metrics (log scale, UNCLIPPED) ===')
for ep, m in ext_metrics.items():
    print(f'  {ep:25s}: MAE={m["MAE"]:.4f}  RAE={m["RAE"]:.4f}  R2={m["R2"]:.4f}  rho={m["Spearman"]:.4f}  tau={m["Kendall"]:.4f}')
print(f'  Macro RAE: {ext_rae:.4f}')

# Clip predictions to train range ± DELTA
for i, ep in enumerate(ENDPOINTS):
    rng_ep = train_maxs[ep] - train_mins[ep]
    lo = train_mins[ep] - DELTA * rng_ep
    hi = train_maxs[ep] + DELTA * rng_ep
    ext_preds[:, i] = np.clip(ext_preds[:, i], lo, hi)

# Compute and print clipped metrics
print('\n=== External test metrics (log scale, CLIPPED) ===')
clipped_metrics = {}
clipped_raes = []
for i, ep in enumerate(ENDPOINTS):
    m = ext_masks[:, i].astype(bool) & np.isfinite(ext_ys[:, i])
    if m.sum() < 3:
        clipped_metrics[ep] = {'MAE': np.nan, 'RAE': np.nan, 'R2': np.nan, 'Spearman': np.nan, 'Kendall': np.nan}
        continue
    clipped_metrics[ep] = official_metrics(ext_ys[m, i], ext_preds[m, i])
    clipped_raes.append(clipped_metrics[ep]['RAE'])
clipped_macro_rae = float(np.nanmean(clipped_raes)) if clipped_raes else np.inf

for ep, m in clipped_metrics.items():
    print(f'  {ep:25s}: MAE={m["MAE"]:.4f}  RAE={m["RAE"]:.4f}  R2={m["R2"]:.4f}  rho={m["Spearman"]:.4f}  tau={m["Kendall"]:.4f}')
print(f'  Macro RAE: {clipped_macro_rae:.4f}')

# Build predictions DataFrame (log scale)
pred_df = pd.DataFrame()
pred_df['SMILES'] = log_test_df['SMILES'].values
pred_df['Molecule Name'] = log_test_df['Molecule Name'].values
for i, ep in enumerate(ENDPOINTS):
    pred_df[ep] = ext_preds[:, i]

# Save predictions
output_dir = Path(f'../experiments/{EXPERIMENT_NAME}')
output_dir.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(output_dir / 'predictions_log_scale.csv', index=False)
print(f'\nPredictions saved to {output_dir / "predictions_log_scale.csv"}')

=== External test metrics (log scale, UNCLIPPED) ===
  LogD                     : MAE=0.4199  RAE=0.5173  R2=0.6502  rho=0.8375  tau=0.6769
  Macro RAE: 0.5173

=== External test metrics (log scale, CLIPPED) ===
  LogD                     : MAE=0.4197  RAE=0.5170  R2=0.6536  rho=0.8375  tau=0.6769
  Macro RAE: 0.5170

Predictions saved to ../experiments/st_random/predictions_log_scale.csv


In [17]:
# === SUMMARY ===

print(f'Experiment: {EXPERIMENT_NAME}')
print(f'NUM_TASKS: {NUM_TASKS}')
print(f'Pretrained: {PRETRAINED_CHECKPOINT is not None}')
print(f'Best epoch: {best_epoch}')
print(f'Best val macro RAE: {best_macro_rae:.4f}')
print()

# Summary table
rows = []
for ep in ENDPOINTS:
    vm = val_metrics.get(ep, {})
    em = ext_metrics.get(ep, {})
    rows.append({
        'Endpoint': ep,
        'Val MAE': vm.get('MAE', np.nan),
        'Val RAE': vm.get('RAE', np.nan),
        'Val R2': vm.get('R2', np.nan),
        'Ext MAE': em.get('MAE', np.nan),
        'Ext RAE': em.get('RAE', np.nan),
        'Ext R2': em.get('R2', np.nan),
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False, float_format='%.4f'))

# Save training history
hist_df = pd.DataFrame(history)
hist_df.to_csv(output_dir / 'training_history.csv', index=False)
print(f'\nTraining history saved to {output_dir / "training_history.csv"}')


Experiment: st_random
NUM_TASKS: 1
Pretrained: False
Best epoch: 1286
Best val macro RAE: 0.2698

Endpoint  Val MAE  Val RAE  Val R2  Ext MAE  Ext RAE  Ext R2
    LogD   0.2511   0.2698  0.9120   0.4199   0.5173  0.6502

Training history saved to ../experiments/st_random/training_history.csv
